In [1]:
using Pkg
Pkg.activate("./")
Pkg.develop(path="../../")
ENV["TAMBOSIM_PATH"] = realpath("../../")

  Activating project at `~/research/TAMBO-MC/notebooks/create_geometry`
   Resolving package versions...
    Updating `~/research/TAMBO-MC/notebooks/create_geometry/Project.toml`
  [37e2e46d] ↑ LinearAlgebra v1.11.0 ⇒ v1.12.0
    Updating `~/research/TAMBO-MC/notebooks/create_geometry/Manifest.toml`
  [c8ffd9c3] - MbedTLS_jll v2.28.6+0
  [f43a241f] ↑ Downloads v1.6.0 ⇒ v1.7.0
  [ac6e5ff7] + JuliaSyntaxHighlighting v1.12.0
  [37e2e46d] ↑ LinearAlgebra v1.11.0 ⇒ v1.12.0
  [ca575930] ↑ NetworkOptions v1.2.0 ⇒ v1.3.0
  [44cfe95a] ↑ Pkg v1.11.0 ⇒ v1.12.0
  [2f01184e] ↑ SparseArrays v1.11.0 ⇒ v1.12.0
  [e66e0078] ↑ CompilerSupportLibraries_jll v1.1.1+0 ⇒ v1.3.0+1
  [deac9b47] ↑ LibCURL_jll v8.6.0+0 ⇒ v8.15.0+0
  [e37daf67] ↑ LibGit2_jll v1.7.2+0 ⇒ v1.9.0+0
  [29816b5a] ↑ LibSSH2_jll v1.11.0+1 ⇒ v1.11.3+1
  [14a3606d] ↑ MozillaCACerts_jll v2023.12.12 ⇒ v2025.5.20
  [4536629a] ↑ OpenBLAS_jll v0.3.27+1 ⇒ v0.3.29+0
  [05823500] ↑ OpenLibm_jll v0.8.1+2 ⇒ v0.8.7+0
  [458c3c95] ~ OpenSSL_jll v3.5.4

"/Users/jlazar/research/TAMBO-MC"

In [3]:
Pkg.add("ProgressMeter")

   Resolving package versions...
    Updating `~/research/TAMBO-MC/notebooks/create_geometry/Project.toml`
  [92933f4c] + ProgressMeter v1.11.0
    Manifest No packages added to or removed from `~/research/TAMBO-MC/notebooks/create_geometry/Manifest.toml`
Precompiling packages...
    713.5 ms  ✓ PrecompileTools
   1237.0 ms  ✓ LAPACK32_jll
   3915.4 ms  ✓ GMT_jll
  63320.6 ms  ✓ GMT
  13600.3 ms  ✓ GMT → GMTParkerFFTExt
  5 dependencies successfully precompiled in 85 seconds. 407 already precompiled.
  1 dependency precompiled but a different version is currently loaded. Restart julia to access the new version. Otherwise, loading dependents of this package may trigger further precompilation to work with the unexpected version.


In [ ]:
using CairoMakie
using Dierckx
using HDF5
using LinearAlgebra
using Makie
using ProgressMeter
using Rotations
using Tambo
using Unitful

include("../plotting_boilerplate.jl")

  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.
[ Info: Precompiling DistributedExt [075b2bf6-2807-5c02-93c4-9c66b4782582] 

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


# Load up the base triangulation and rotate to new frame
## First load up the vertices of the previous triangulation

In [ ]:
filename, groupname = "./triangulation.h5", "base_triangulation_30000"

original_vertices = h5open(filename) do file
    group = file[groupname]
    deg2rad.(read(group["vertices"]))
end

## Define some geometry helpers for rotating

In [ ]:
longlat0 = deg2rad(-72.279397), deg2rad(-15.622267)

axis = cross([0, 0, 1], Tambo.longlat_to_cart(longlat0...))
angle = acos(dot([0, 0, 1], Tambo.longlat_to_cart(longlat0...)))
rotation = AngleAxis(angle, axis...)

## Rotate all the points to be centered around the new point of interest

In [ ]:
rotated_points = [rotation * Tambo.longlat_to_cart(original_vertices[idx, :]...) for idx in 1:size(original_vertices)[1]]
rotated_longlats = [Tambo.cart_to_longlat(x...) for x in rotated_points]

vertices = zeros(size(original_vertices))
for idx in 1:length(rotated_longlats)
    vertices[idx, :] = rotated_longlats[idx]
end

fig = Figure()
ax = Axis(
    fig[1, 1],
    xlabel="Longitude [deg.]",
    ylabel="Latitude [deg.]"
)

scatter!(
    ax,
    rad2deg.(vertices[:, 1]),
    rad2deg.(vertices[:, 2]),
    alpha=0.2,
    markersize=3
)

fig

# Make topography from previously computed triangulation

This assumes that you have run the other notebook first and have downloaded `earth_relief_01m.grd` from `http://oceania.generic-mapping-tools.org/` and have placed it in this directory. We're also going to write this to an `HDF5` file so that we can access it bit by bit, without readin the whole thing into memory, which is a pain.

## Read the `.grd` file and save to `HDF5`
I've commented this block for now because it can take a very long time to run and should only be done once.

In [ ]:
# using GMT: gmtread
# g = gmtread("./earth_relief_15s.grd");
# h5open("earth_relief_15s.h5", "w") do file
#     file["longitude"] = Float32.(g.x)
#     file["latitude"] = Float32.(g.y)
#     file["elevation"] = g.z
# end

## Load the longitude and latitude points

In [ ]:
longs, lats = h5open("earth_relief_15s.h5") do file
    longs, lats = deg2rad.(file["longitude"][1:end-1]), deg2rad.(file["latitude"][1:end-1])
end

## Compute the spline for subsamples of the Earth
The elevation map has too many points to spline all at once so we have to split it up into subsections.
We have chosen to do do this in slices of longitude for simplicity.
Below I have some helper functions for this task.

In [ ]:
function find_indexes(
    long1::Real,
    long2::Real,
    longs::Vector,
    padding::Real,
    branch::Tuple{Real, Real}=(-π, π)
)::Tuple{Vector{Int}, Vector{Float64}}
    @assert all(diff(longs) .> 0) "Longitudes not sorted"
    @assert branch[2] - branch[1]==2π "Branch not full circle"
    @assert (branch[1] <= long1 <= branch[2]) && (branch[1] <= long2 <= branch[2])
    longmin, longmax = long1 - padding, long2 + padding
    long_idxs, spl_longs = Int[], Float32[]
    if longmin < branch[1]
        idxs = findall(2π + longmin .< longs)
        long_idxs = vcat(long_idxs, idxs)
        spl_longs = vcat(spl_longs, longs[idxs] .- 2π)
    end
    idxs = findall(longmin .<= longs .< longmax)
    if length(idxs) > 0
        long_idxs = vcat(long_idxs, idxs)
        spl_longs = vcat(spl_longs, longs[idxs])
    end
    if branch[2] < longmax
        idxs = findall(longs .< longmax - 2π)
        long_idxs = vcat(long_idxs, idxs)
        spl_longs = vcat(spl_longs, longs[idxs] .+ 2π)
    end
    return long_idxs, spl_longs
end

In [ ]:
function split_to_contiguous_ranges(idxs::Vector{Int})::Vector{AbstractUnitRange}
    ranges = UnitRange[]    
    l = first(idxs)
    for (cur, next) in zip(idxs, idxs[2:end])
        if next==cur+1
            continue
        end
        push!(ranges, l:cur)
        l = next
    end
    push!(ranges, l:last(idxs))
    return ranges
end

# Compute the elevation for each point
The datafile we're using has the depths of the oceans at every point(!!?), but we are going to wrap the PREM in a shell and so we are going to ignore all values below sea level.
I also introduce a hack to make sure the shell is one centimeter off the PREM to avoid issues when ray tracing.

In [ ]:
degree = 2
step = deg2rad(5)
padding = step / 20
we = -π:step:π

vertex_elevations = zeros(size(vertices, 1)) .* u"m"

@showprogress for (long1, long2) in zip(we, we[2:end])
    
    long_idxs, spl_longs = find_indexes(long1, long2, longs, padding)
    ranges = split_to_contiguous_ranges(long_idxs)
    
    elevs = h5open("./earth_relief_15s.h5") do file
        elevs = nothing
        for range in ranges
            if elevs==nothing
                elevs = file["elevation"][1:length(lats), range]
            else
                elevs = hcat(elevs, file["elevation"][1:length(lats), range])
            end
        end
        elevs
    end

    itp = Spline2D(spl_longs, lats, elevs'; kx=degree, ky=degree, s=0.0)
    
    idxs = long1 .< vertices[:, 1] .<= long2
    
    vertex_elevations[idxs] = map(x->maximum([itp(x...), 1]) * u"m", eachrow(vertices)[idxs])

end

vertex_elevations

In [ ]:
fig = Figure()
ax = Axis(
    fig[1, 1],
    xlabel=L"\phi_{\mathrm{long}}~\left[^{\circ}\right]",
    ylabel=L"\sin(\theta_{\mathrm{lat}})"
)

sc = scatter!(
    ax,
    rad2deg.(vertices[:, 1]),
    sin.(vertices[:, 2]),
    color=ustrip.(vertex_elevations),
    alpha=0.1
)

cbar = Colorbar(fig[1, 2], sc, label="Elevation [m]")

fig

# Convert these to trinagles in 3D space

In [ ]:
faces = h5open(filename) do file
    group = file[groupname]
    faces = read(group["faces"])
end;

In [ ]:
# PREM model https://lweb.cfa.harvard.edu/~lzeng/papers/PREM.pdf
radii = [1221.5, 3480.0, 3630.0, 5600.0, 5701.0, 5771.0, 5971.0, 6151.0, 6291.0, 6346.6, 6356.0, 6368.0, 6371.0] .* u"km"
# For now I am leaving these blank since they have a variable density and *shouldn't* get
# that deep into the Earth, but I would like it to break if we do. We can replace with average
# density or something if we have to cross that bridge
# densities = [NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, 2.9, 2.6, 2.6] .* u"g"
rearth = radii[end]

vertices_3d = zeros((size(vertices, 1), 3)) * u"m"

MAGIC_NUMBER = 1

for (idx, elevation) in enumerate(vertex_elevations)
    vertices_3d[idx, :] = Tambo.longlat_to_cart(vertices[idx, :]...) * (rearth + MAGIC_NUMBER * elevation)
end

In [ ]:
outkey = replace(groupname, "base_triangulation"=>"colca_valley")
h5open(filename, "r+") do file
    if outkey in keys(file)
        delete_object(file[outkey])
    end
    group = create_group(file, outkey)
    group["location"] = rad2deg.([longlat0[1], longlat0[2]])
    group["radii"] = ustrip.(radii .|> u"m")
    group["faces"] = faces
    group["vertices"] = ustrip.(vertices_3d .|> u"m")
end

# This can be extended to any other elevation profile you want
Let's try to make an idelaized valley.
Note that these numbers are enormous, and are most so that we can see anything in the plot.

In [ ]:
incline = deg2rad(35)
h1 = 2_000
h2 = 500_000
l = (h2 - h1) / tan(incline)

function perfect_valley_fxn(long, lat)
    Δx = abs(lat-longlat0[2])
    return minimum([(h2-h1) * Δx / tan(incline) + 2000, h2])
end

In [ ]:
rearth = 6_371 * u"km"

vertices_3d = Vector{Quantity{Float64, Unitful.𝐋, typeof(u"km")}}[]
for vertex in eachrow(vertices)
    elevation = perfect_valley_fxn(vertex...) * u"m"
    push!(vertices_3d, Tambo.longlat_to_cart(vertex...) * (rearth + elevation))
end

In [ ]:
fig = Figure(size = (600, 600))
ax = Axis3(fig[1,1], azimuth=deg2rad(-70))    

mesh!(
    ax,
    [Point3f(ustrip.(vx)) for vx in vertices_3d],
    faces
)

display(fig)

I guess it's more of a Great Valley, and now it runs west-east instead of north-south, but hopefully we will manage.
It also might make sense to make the valley taper off with a sigmoid or something, but we can experiment with that as it becomes an issue